In [1]:
import random
import numpy as np
import os
import torch 

def set_seed(seed=24):
    """Setea semilla para reproducibilidad general"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    
    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # si usas multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # TensorFlow (Descomenta si lo necesitas)
    # tf.random.set_seed(seed)
    
    print(f"Semilla fijada en: {seed}")

# Llamar a la función
set_seed(24)

Semilla fijada en: 24


In [2]:
import sys
sys.path.append('../')
 
import pandas as pd 
from sklearn.metrics import cohen_kappa_score, accuracy_score,balanced_accuracy_score
from plotly import express as px
from tutoriales.utils import plot_confusion_matrix, get_artifact_filename
from json import loads
from joblib import load, dump
import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact
from optuna.visualization import plot_contour
from optuna.visualization import plot_param_importances

d:\anaconda3\envs\ldi2_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sys, numpy.core
# Shim: permite cargar joblib guardados con numpy >= 2.0 en entornos con numpy 1.x
sys.modules.setdefault("numpy._core", numpy.core)
for _sub in ["numeric", "multiarray", "umath", "fromnumeric", "arrayprint", "strings"]:
    mod = getattr(numpy.core, _sub, numpy.core)
    sys.modules.setdefault(f"numpy._core.{_sub}", mod)


In [39]:
# Paths
BASE_DIR = '../'
PATH_TO_TRAIN = os.path.join(BASE_DIR, "work/cleaned/train_clean.csv")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")

In [20]:
#study_lgb = optuna.create_study(direction='maximize',
#                            storage="sqlite:///../work/db-delfi.sqlite3",  # Specify the storage URL here.
#                            study_name="lgbm_modelo_2__variables_completas",
#                            load_if_exists = True)



In [21]:
#lgb_dataset = load(os.path.join(PATH_TO_TEMP_FILES,get_artifact_filename(study_lgb,'test')))

In [ ]:
#lgb_dataset

In [40]:
MODEL_NAME = '01 DistilBert'
MODEL_VERSION = '3.0'

study_bert = optuna.create_study(direction='maximize',
                            storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
                            study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
                            load_if_exists = True)

bert_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_bert,'test')))

[I 2026-04-30 17:17:21,337] Using an existing study with name '01 DistilBert_3.0' instead of creating a new one.


In [ ]:
#bert_dataset.shape

In [41]:
# Cargar el modelo ResNet
MODEL_NAME_RESNET = '04 ResNet Augment'
MODEL_VERSION_RESNET = '1.0.0'

study_resnet = optuna.create_study(
    direction='maximize',
    storage="sqlite:///../work/optuna_artifacts/db.sqlite3",
    study_name=f'{MODEL_NAME_RESNET}_{MODEL_VERSION_RESNET}',
    load_if_exists=True
)

[I 2026-04-30 17:17:24,347] Using an existing study with name '04 ResNet Augment_1.0.0' instead of creating a new one.


In [42]:
#resnet_dataset = load(os.path.join(PATH_TO_TEMP_FILES, 'test_04 ResNet Augment_1.0.0_1.joblib'))
resnet_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_resnet,'test')))

In [ ]:
#resnet_dataset.shape

In [43]:
#merged_datasets = lgb_dataset[['PetID', 'pred', 'AdoptionSpeed']].rename({'pred':'lgb_pred_score'},axis=1).merge(bert_dataset[['PetID', 'pred']].rename({'pred':'bert_pred_score'},axis=1),
#                  on='PetID', how='outer')

merged_datasets = resnet_dataset[['PetID', 'pred', 'AdoptionSpeed']].rename({'pred': 'resnet_pred_score'}, axis=1).merge(bert_dataset[['PetID', 'pred']].rename({'pred':'bert_pred_score'},axis=1),
                  on='PetID', how='outer')

In [ ]:
#merged_datasets.shape

In [ ]:
#merged_datasets.head()

In [44]:
print(merged_datasets["resnet_pred_score"].shape[0])
print(merged_datasets["bert_pred_score"].shape[0])

2999
2999


In [ ]:
# Unir ResNet al dataframe fusionado
#merged_datasets = merged_datasets.merge(
#    resnet_dataset[['PetID', 'pred']].rename({'pred': 'resnet_pred_score'}, axis=1),
#    on='PetID', how='outer'
#)

In [ ]:
# Limpiar nulos (rellenar con arrays de ceros si algún modelo no tiene predicción para un PetID)
merged_datasets['resnet_pred_score'] = [np.zeros(5) if type(i) is float else i for i in merged_datasets['resnet_pred_score']]
merged_datasets['bert_pred_score'] = [np.zeros(5) if type(i) is float else i for i in merged_datasets['bert_pred_score']]
#merged_datasets['lgb_pred_score'] = [np.zeros(5) if type(i) is float else i for i in merged_datasets['lgb_pred_score']]

# Eliminar filas con AdoptionSpeed nula
merged_datasets = merged_datasets.dropna(subset=['AdoptionSpeed'])

In [27]:
#merged_datasets.head()
#merged_datasets

In [46]:
all_values = [item for sublist in merged_datasets["resnet_pred_score"] for item in sublist]
min_val = min(all_values)
max_val = max(all_values)

def normalizar_lista(lista):
    return [(x - min_val) / (max_val - min_val) for x in lista]

merged_datasets["resnet_pred_score"] = merged_datasets["resnet_pred_score"].apply(normalizar_lista)

all_values1 = [item for sublist in merged_datasets["bert_pred_score"] for item in sublist]
min_val1 = min(all_values1)
max_val1 = max(all_values1)

def normalizar_lista(lista):
    return [(x - min_val1) / (max_val1 - min_val1) for x in lista]

merged_datasets["bert_pred_score"] = merged_datasets["bert_pred_score"].apply(normalizar_lista)


In [ ]:
#merged_datasets

In [47]:
# Optimización con Optuna para 3 pesos
def objective(trial):
    # Definir pesos para los tres modelos
    #########################w_lgb = trial.suggest_float('w_lgb', 0.0, 1.0)
    w_bert = trial.suggest_float('w_bert', 0.0, 1.0)
    w_resnet = trial.suggest_float('w_resnet', 0.0, 1.0)
    
    # Normalización
    #########################total_w = w_lgb + w_bert + w_resnet
    total_w = w_bert + w_resnet
    
    # Cálculo vectorizado para mayor velocidad
    # Convertimos las columnas de scores en una matriz 3D o sumamos directamente
    #########################lgb_scores = np.stack(merged_datasets['lgb_pred_score'].values)
    bert_scores = np.stack(merged_datasets['bert_pred_score'].values)
    resnet_scores = np.stack(merged_datasets['resnet_pred_score'].values)
    
    combined_scores = (
        #########################(w_lgb / total_w) * lgb_scores + 
        (w_bert / total_w) * bert_scores + 
        (w_resnet / total_w) * resnet_scores
    )
    
    preds_final = np.argmax(combined_scores, axis=1)
    
    return cohen_kappa_score(merged_datasets['AdoptionSpeed'], preds_final, weights='quadratic')

In [48]:
# Ejecutar el estudio
STORAGE_URL = "sqlite:///../work/db-blend.sqlite3"
study_blend = optuna.create_study(
    direction='maximize',
    storage=STORAGE_URL,
    study_name="Ensemble_LGB_BERT_ResNet",
    load_if_exists=True
)
study_blend.optimize(objective, n_trials=100)

[I 2026-04-30 17:17:44,781] Using an existing study with name 'Ensemble_LGB_BERT_ResNet' instead of creating a new one.
[I 2026-04-30 17:17:45,118] Trial 903 finished with value: 0.2862306302149422 and parameters: {'w_bert': 0.8274837471699709, 'w_resnet': 0.9650316933203071}. Best is trial 151 with value: 0.35228696324248765.
[I 2026-04-30 17:17:45,159] Trial 904 finished with value: 0.2847461892621028 and parameters: {'w_bert': 0.8000717728861293, 'w_resnet': 0.8668606398789853}. Best is trial 151 with value: 0.35228696324248765.
[I 2026-04-30 17:17:45,205] Trial 905 finished with value: 0.28499373279606843 and parameters: {'w_bert': 0.73897060142052, 'w_resnet': 0.9316819986235156}. Best is trial 151 with value: 0.35228696324248765.
[I 2026-04-30 17:17:45,252] Trial 906 finished with value: 0.2853492097295396 and parameters: {'w_bert': 0.7731028282418559, 'w_resnet': 0.9995262061324679}. Best is trial 151 with value: 0.35228696324248765.
[I 2026-04-30 17:17:45,293] Trial 907 finishe

In [ ]:
# Resultados
best_params = study_blend.best_params
sum_best_w = sum(best_params.values())

print(f"Mejor Kappa: {study_blend.best_value:.4f}")
print(f"Pesos óptimos: {best_params}")

In [ ]:
# Crear la columna de predicción final optimizada

def _prediction_vector(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        return np.asarray(value, dtype=float)
    return np.zeros(5, dtype=float)

merged_datasets['blend_pred_score'] = [
    (best_params['w_bert'] / sum_best_w) * _prediction_vector(r['bert_pred_score']) +
    (best_params['w_resnet'] / sum_best_w) * _prediction_vector(r['resnet_pred_score'])
    for _, r in merged_datasets.iterrows()
]

In [ ]:


# Generar el gráfico de importancia de hiperparámetros (pesos)
fig = plot_param_importances(study_blend)

fig.update_layout(
    title="Importancia de los Modelos en el Ensamble",
    xaxis_title="Importancia relativa",
    yaxis_title="Modelo (Pesos)",
    template="plotly_white"
)

fig.show()


In [ ]:
##################fig_contour = plot_contour(study_blend, params=['w_lgb', 'w_bert', 'w_resnet'])
fig_contour = plot_contour(study_blend, params=['w_bert', 'w_resnet'])

fig_contour.update_layout(
    title="Interacción de Pesos y Rendimiento (Kappa)",
    width=900,
    height=800
)

fig_contour.show()

In [ ]:
# Guardar el resultado final en un archivo temporal y subirlo a Optuna
final_filename = "merged_predictions_optimized.joblib"
dump(merged_datasets, final_filename)


In [ ]:
# Subir el archivo como artefacto del mejor trial
#artifact_store = FileSystemArtifactStore(base_path=PATH_TO_OPTUNA_ARTIFACTS)
#upload_artifact(
#    trial=study_blend.best_trial, 
#    file_path=final_filename, 
#    artifact_store=artifact_store
#)


In [ ]:
#-------------------------------------------------------------

In [ ]:
#merged_datasets

In [ ]:
#merged_datasets['blend_pred_score'] = [r['lgb_pred_score']+r['bert_pred_score'] for i,r in merged_datasets.iterrows()]

In [ ]:
#merged_datasets['lgb_pred_score']

In [ ]:
#merged_datasets['lgb_pred'] = [r.argmax() for r in merged_datasets['lgb_pred_score']]
#merged_datasets['bert_pred'] = [r.argmax() for r in merged_datasets['bert_pred_score']]
#merged_datasets['blended_pred'] = [r.argmax() for r in merged_datasets['blend_pred_score']]

In [ ]:
#plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
#                      merged_datasets['lgb_pred'], 
#                    title = 'LGB Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
#                                                                    merged_datasets['lgb_pred'], 
#                                                                    weights='quadratic')))

In [ ]:
#plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
#                      merged_datasets['bert_pred'], 
#                    title = 'Bert Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
#                                                                   merged_datasets['bert_pred'], 
#                                                                    weights='quadratic')))



In [ ]:
#plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
#                     merged_datasets['blended_pred'], 
#                    title = 'Blended Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
#                                                                    merged_datasets['blended_pred'], 
#                                                                    weights='quadratic')))
